## 관세청_품목별 국가별 수출입실적(GW)

In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from src.config import get_customs_api_key

#### 요청 메세지
    - serviceKey: 인증키
    - strtYymm: 시작년월(ex. 202504)
    - endYymm: 종료년월(ex. 202604) (시작년월과 종료년월 사이의 조회기간은 최대 1년)
    - hsSgn: 품목코드
    - cntyCd: 국가코드
#### 응답 메세지
    - resultCode: 결과코드
    - resultMsg: 결과메세지
    - year: 기간
    - statCdCntnKor1: 국가명
    - statCd: 국가코드
    - statKor: 품목명
    - hsCd: HS코드
    - expWgt: 수출중량(kg)
    - expDlr: 수출금액(달러)
    - impWgt: 수입중량(kg)
    - impDlr: 수입금액(달러)
    - balPayments: 무역수지(달러)


In [ ]:
url = "https://apis.data.go.kr/1220000/nitemtrade/getNitemtradeList"
serviceKey = get_customs_api_key()

In [ ]:
strtYymm = "201502"
endYymm = "201601"
hsSgn = "1001999090"
cntyCd = "US"

In [ ]:
params = {"serviceKey": serviceKey,
          "strtYymm": strtYymm,
          "endYymm": endYymm,
          "hsSgn": hsSgn,
          "cntyCd": cntyCd,}

res = requests.get(url, params=params)

In [ ]:
root = ET.fromstring(res.text)

result_code = root.findtext("./header/resultCode")
result_msg = root.findtext("./header/resultMsg")
print("API 결과:", result_code, result_msg)

rows = []
for item in root.findall("./body/items/item"):
    if item.findtext("year") == "총계":
            continue

    row = {
        "year": item.findtext("year"),
        "country": item.findtext("statCdCntnKor1"),
        "country_cd": item.findtext("statCd"),
        "item_name": item.findtext("statKor"),
        "hs_cd": item.findtext("hsCd"),
        "imp_dlr": int(item.findtext("impDlr")),
        "exp_dlr": int(item.findtext("expDlr")),
        "imp_wgt": int(item.findtext("impWgt")),
        "exp_wgt": int(item.findtext("expWgt")),
        "balance": int(item.findtext("balPayments")),
    }

    rows.append(row)

df = pd.DataFrame(rows)

API 결과: 00 정상서비스.


In [ ]:
df

,year,country,country_cd,item_name,hs_cd,imp_dlr,exp_dlr,imp_wgt,exp_wgt,balance
0,2015.02,미국,US,기타,1001999090,337664,0,1096728,0,-337664
1,2015.03,미국,US,기타,1001999090,1316648,0,4187638,0,-1316648
2,2015.04,미국,US,기타,1001999090,238826,0,742555,0,-238826
3,2015.05,미국,US,기타,1001999090,314245,0,961758,0,-314245
4,2015.06,미국,US,기타,1001999090,734144,0,2119230,0,-734144
5,2015.07,미국,US,기타,1001999090,1314484,65,4171509,16,-1314419
6,2015.08,미국,US,기타,1001999090,48127,0,10036,0,-48127
7,2015.09,미국,US,기타,1001999090,520,0,52,0,-520
8,2015.10,미국,US,기타,1001999090,88328,83,319505,4,-88245
9,2015.11,미국,US,기타,1001999090,3301967,0,11651153,0,-3301967


### 함수로...
기간이 1년이 넘으면 자동으로 분할 + 원하는 컬럼 입력

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime

def get_trade_data(
    service_key,
    start,
    end,
    hs_code,
    country,
    columns=None,
    rename_map=None,
    exclude_total=True
):
    """
    관세청 품목별 국가별 수출입실적 조회

    Parameters
    ----------
    service_key : str
        공공데이터포털 Decoding 키
    start : str
        시작년월 (YYYYMM)
    end : str
        종료년월 (YYYYMM)
    hs_code : str
        품목코드
    country : str
        국가코드
    columns : list[str], optional
        가져올 XML 태그 목록
        기본값:
        ["year", "statCdCntnKor1", "statCd", "statKor", "hsCd",
         "impDlr", "expDlr", "impWgt", "expWgt", "balPayments"]
    rename_map : dict, optional
        컬럼명 변경용 매핑
        예: {"statCdCntnKor1": "country", "impDlr": "imp_dlr"}
    exclude_total : bool, default True
        year == "총계" 행 제외 여부

    Returns
    -------
    pandas.DataFrame
    """

    if columns is None:
        columns = [
            "year", "statCdCntnKor1", "statCd", "statKor", "hsCd",
            "impDlr", "expDlr", "impWgt", "expWgt", "balPayments"
        ]

    if rename_map is None:
        rename_map = {}

    url = "https://apis.data.go.kr/1220000/nitemtrade/getNitemtradeList"

    numeric_columns = {"impDlr", "expDlr", "impWgt", "expWgt", "balPayments"}

    def validate_yymm(yymm):
        try:
            datetime.strptime(yymm, "%Y%m")
        except ValueError as e:
            raise ValueError(f"날짜 형식이 잘못되었습니다: {yymm}. YYYYMM 형식으로 넣어주세요.") from e

    def add_months(dt, months):
        year = dt.year + (dt.month - 1 + months) // 12
        month = (dt.month - 1 + months) % 12 + 1
        return dt.replace(year=year, month=month)

    def split_period(start_yymm, end_yymm):
        """
        조회기간이 1년을 넘으면 자동 분할.
        한 요청당 최대 12개월.
        예:
        201501 ~ 201612
        -> [('201501','201512'), ('201601','201612')]
        """
        periods = []

        start_dt = datetime.strptime(start_yymm, "%Y%m")
        end_dt = datetime.strptime(end_yymm, "%Y%m")

        current_start = start_dt

        while current_start <= end_dt:
            # 시작월 포함 최대 12개월 => +11개월
            current_end = add_months(current_start, 11)

            if current_end > end_dt:
                current_end = end_dt

            periods.append((
                current_start.strftime("%Y%m"),
                current_end.strftime("%Y%m")
            ))

            current_start = add_months(current_end, 1)

        return periods

    validate_yymm(start)
    validate_yymm(end)

    start_dt = datetime.strptime(start, "%Y%m")
    end_dt = datetime.strptime(end, "%Y%m")

    if start_dt > end_dt:
        raise ValueError("start는 end보다 클 수 없습니다.")

    periods = split_period(start, end)
    all_rows = []

    for s, e in periods:
        params = {
            "serviceKey": service_key,
            "strtYymm": s,
            "endYymm": e,
            "hsSgn": hs_code,
            "cntyCd": country,
        }

        res = requests.get(url, params=params, timeout=30)

        if res.status_code != 200:
            raise RuntimeError(
                f"HTTP 오류: {res.status_code}\n"
                f"응답 내용: {res.text[:300]}"
            )

        try:
            root = ET.fromstring(res.text)
        except ET.ParseError as e:
            raise RuntimeError(
                f"XML 파싱 실패.\n응답 내용: {res.text[:300]}"
            ) from e

        result_code = root.findtext("./header/resultCode")
        result_msg = root.findtext("./header/resultMsg")

        if result_code != "00":
            raise RuntimeError(
                f"API 오류: resultCode={result_code}, resultMsg={result_msg}"
            )

        items = root.findall("./body/items/item")

        for item in items:
            if exclude_total and item.findtext("year") == "총계":
                continue

            row = {}

            for col in columns:
                value = item.findtext(col)

                if col in numeric_columns and value not in (None, "", "-"):
                    try:
                        value = int(value)
                    except ValueError:
                        pass

                row[col] = value

            all_rows.append(row)

    df = pd.DataFrame(all_rows)

    if rename_map:
        df = df.rename(columns=rename_map)

    return df

In [ ]:
service_key = get_customs_api_key()
start = "202405"
end = "202604"
hs_code = "1001999090"
country = "US"

df = get_trade_data(service_key, start, end, hs_code, country)

In [ ]:
df

,year,statCdCntnKor1,statCd,statKor,hsCd,impDlr,expDlr,impWgt,expWgt,balPayments
0,2024.05,미국,US,기타,1001999090,302163,0,886799,0,-302163
1,2024.06,미국,US,기타,1001999090,1409922,0,4401086,0,-1409922
2,2024.07,미국,US,기타,1001999090,1037476,0,3601724,0,-1037476
3,2024.08,미국,US,기타,1001999090,676646,0,2497962,0,-676646
4,2024.09,미국,US,기타,1001999090,144650,0,446118,0,-144650
5,2024.10,미국,US,기타,1001999090,904125,0,2534514,0,-904125
6,2024.11,미국,US,기타,1001999090,196420,0,545489,0,-196420
7,2024.12,미국,US,기타,1001999090,112,0,8,0,-112
8,2025.01,미국,US,기타,1001999090,30231,0,83794,0,-30231
9,2025.02,미국,US,기타,1001999090,1140328,0,4107346,0,-1140328
